In [1]:
# Section 1: Imports

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, round as spark_round, max as spark_max
from pyspark.sql.types import FloatType
from google.colab import files

In [2]:
# Section 2: Upload File

uploaded = files.upload()

if not uploaded:
    raise Exception("No file uploaded. Please upload a CSV file.")

file_name = next(iter(uploaded))
print(f"Local file uploaded successfully: {file_name}")


Saving Flight Dataset - CSV(in).csv to Flight Dataset - CSV(in).csv
Local file uploaded successfully: Flight Dataset - CSV(in).csv


In [3]:
# Section 3: Create Spark session and load CSV

spark = SparkSession.builder.appName("Domestic Flight Delay Analysis").getOrCreate()

flights_df = spark.read.csv(file_name, header=True, inferSchema=True)

flights_df.printSchema()
flights_df.show(5)


root
 |-- FL_DATE: string (nullable = true)
 |-- DEP_DELAY: integer (nullable = true)
 |-- ARR_DELAY: integer (nullable = true)
 |-- AIR_TIME: integer (nullable = true)
 |-- DISTANCE: integer (nullable = true)
 |-- DEP_TIME: double (nullable = true)
 |-- ARR_TIME: double (nullable = true)

+--------+---------+---------+--------+--------+---------+---------+
| FL_DATE|DEP_DELAY|ARR_DELAY|AIR_TIME|DISTANCE| DEP_TIME| ARR_TIME|
+--------+---------+---------+--------+--------+---------+---------+
|1/1/2006|        5|       19|     350|    2475| 9.083333|12.483334|
|1/2/2006|      167|      216|     343|    2475|11.783334|15.766666|
|1/3/2006|       -7|       -2|     344|    2475| 8.883333|12.133333|
|1/4/2006|       -5|      -13|     331|    2475| 8.916667|    11.95|
|1/5/2006|       -3|      -17|     321|    2475|     8.95|11.883333|
+--------+---------+---------+--------+--------+---------+---------+
only showing top 5 rows


In [4]:
# Section 4: Data Cleaning & Standardization

from pyspark.sql.types import FloatType

numeric_cols = ["DEP_DELAY", "ARR_DELAY", "AIR_TIME", "DISTANCE", "DEP_TIME", "ARR_TIME"]

for col_name in numeric_cols:
    flights_df = flights_df.withColumn(col_name, col(col_name).cast(FloatType()))

flights_df = flights_df.dropna(subset=numeric_cols)


In [5]:
# Section 5: Task 1 – Flights that arrived earlier than expected

def flights_arrived_early(df):
    return df.filter(col("ARR_DELAY") < 0).count()

print("Flights arrived earlier than expected:", flights_arrived_early(flights_df))


Flights arrived earlier than expected: 534655


In [6]:
# Section 6: Task 2 – Typical departure time for flights over 2000 miles

def typical_departure_time_long_flights(df, distance_threshold=2000):
    return df.filter(col("DISTANCE") > distance_threshold)\
             .agg(spark_round(avg("DEP_TIME"), 2))\
             .collect()[0][0]

print("Typical departure time for flights >2000 miles:", typical_departure_time_long_flights(flights_df))


Typical departure time for flights >2000 miles: 13.97


In [7]:
# Section 7: Task 3 – Proportion of flights with arrival delays > 60 minutes

def proportion_long_arrival_delays(df, delay_threshold=60):
    total_flights = df.count()
    delayed_flights = df.filter(col("ARR_DELAY") > delay_threshold).count()
    return round(delayed_flights / total_flights, 4)

print("Proportion of flights with ARR_DELAY > 60 mins:", proportion_long_arrival_delays(flights_df))


Proportion of flights with ARR_DELAY > 60 mins: 0.0531


In [8]:
# Section 8: Task 4 – Average airtime for flights departing before 9:00 am

def avg_airtime_early_departures(df, cutoff_time=9.0):
    return df.filter(col("DEP_TIME") < cutoff_time)\
             .agg(spark_round(avg("AIR_TIME"), 2))\
             .collect()[0][0]

print("Average airtime for flights departing before 9:00 am:", avg_airtime_early_departures(flights_df))


Average airtime for flights departing before 9:00 am: 111.36


In [9]:
# Section 9: Task 5 – Maximum arrival delay for flights with no departure delay

def max_arrival_delay_no_departure_delay(df):
    return df.filter(col("DEP_DELAY") <= 0)\
             .agg(spark_max("ARR_DELAY"))\
             .collect()[0][0]

print("Maximum arrival delay for flights with no departure delay:", max_arrival_delay_no_departure_delay(flights_df))


Maximum arrival delay for flights with no departure delay: 701.0
